In [4]:
import time
import pandas as pd
from urllib.parse import quote_plus

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# ⚙ 설정값
# =========================

INPUT_EXCEL = "category.xlsx"                # 소분류 목록 엑셀
OUTPUT_EXCEL = "babitalk_event_urls.xlsx"    # 결과 저장 파일

COL_BIG = "대분류"
COL_MID = "중분류"
COL_SMALL = "소분류"                         # 검색에 사용할 컬럼

BABITALK_SEARCH_BASE = "https://web.babitalk.com/search"

PAGE_LOAD_TIMEOUT = 30
WAIT_AFTER_LOAD = 1.5
HEADLESS = False

SCROLL_MAX_STEP = 80         # 최대 스크롤 횟수
SCROLL_SLEEP = 0.5           # 스크롤 간 대기 시간(짧게)
NO_GAIN_LIMIT = 6            # 카드 수 증가 없을 때 중단 기준

MAX_CARDS_PER_KEYWORD = 80   # 한 키워드당 최대 수집 개수
MAX_SECONDS_PER_KEYWORD = 25 # 한 키워드당 최대 수행 시간(초)


# =========================
# 공통 유틸
# =========================

def setup_driver(headless: bool = False):
    options = Options()
    options.add_argument("--window-size=1400,900")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-blink-features=AutomationControlled")
    if headless:
        options.add_argument("--headless=new")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    driver.set_script_timeout(PAGE_LOAD_TIMEOUT)
    return driver


def is_no_result_page_babitalk(driver) -> bool:
    """
    바비톡 검색에서 결과가 없을 때 등장하는 문구를 기준으로
    실제 화면에 표시되는 경우만 True 반환.
    """
    try:
        elems = driver.find_elements(
            By.XPATH,
            "//*[contains(text(), '검색된 이벤트가 없어요') "
            "or contains(text(), '결과가 없습니다')]"
        )
        for el in elems:
            try:
                if el.is_displayed():
                    return True
            except Exception:
                continue
    except Exception:
        pass
    return False


def collect_babitalk_event_cards(driver, keyword: str):
    """
    바비톡 검색 결과 페이지에서:
      - #content-body(스크롤 컨테이너)를 스크롤하면서
      - div[data-event-id] 카드들을 전부 모으고
      - 카드 텍스트에 keyword(공백 제거)가 포함된 것만 반환.

    중간에 다음 조건을 만족하면 바로 종료:
      - 카드가 전혀 안 보이는 상태가 2회 연속
      - 새 카드가 더 이상 늘어나지 않는 상태가 NO_GAIN_LIMIT 회
      - 수집 카드 개수가 MAX_CARDS_PER_KEYWORD 이상
      - 경과 시간이 MAX_SECONDS_PER_KEYWORD 초 이상
    """
    kw = keyword.replace(" ", "")
    start_time = time.time()

    def current_cards():
        cards = driver.execute_script("""
            const nodes = Array.from(
                document.querySelectorAll('div[data-event-id]')
            );
            const results = [];
            const seen = new Set();

            for (const el of nodes) {
                const id = el.getAttribute('data-event-id');
                if (!id || seen.has(id)) continue;
                seen.add(id);

                const text = el.innerText || '';
                results.push({ id, text });
            }
            return results;
        """)
        return cards or []

    collected = {}   # event_id -> text
    last_count = 0
    no_gain = 0
    empty_rounds = 0  # 카드가 1개도 안 보이는 턴 수

    for step in range(SCROLL_MAX_STEP):
        # ⏱ 시간 제한 체크
        elapsed = time.time() - start_time
        if elapsed >= MAX_SECONDS_PER_KEYWORD:
            print(f"   ⏱ {MAX_SECONDS_PER_KEYWORD}초 경과 → 강제 종료")
            break

        cards = current_cards()
        for c in cards:
            eid = c.get("id")
            text = c.get("text", "")
            if not eid:
                continue
            collected[eid] = text

        cur = len(collected)
        gain = cur - last_count
        print(f"   ▸ Scroll {step+1:03d} / cards: {cur} (+{gain})")

        # 새 카드 증가 여부
        if gain == 0:
            no_gain += 1
        else:
            no_gain = 0

        # 카드 존재 여부 (0개 상태가 계속되는지)
        if cur == 0:
            empty_rounds += 1
        else:
            empty_rounds = 0

        last_count = cur

        # 조건 1) 새 카드가 안 늘어나는 경우
        if no_gain >= NO_GAIN_LIMIT:
            print("   ✅ 더 이상 새 카드 없음 → 스크롤 중단")
            break

        # 조건 2) 카드가 전혀 안 보이는 경우 2번만 확인 후 포기
        if empty_rounds >= 2:
            print("   ⚪ 카드가 전혀 안 보임 → 2회 확인 후 스킵")
            break

        # 조건 3) 카드가 너무 많을 때 상한선 도달
        if cur >= MAX_CARDS_PER_KEYWORD:
            print(f"   🎯 카드 {MAX_CARDS_PER_KEYWORD}개 수집 완료 → 중단")
            break

        # 실제 스크롤 동작
        driver.execute_script("""
            const scroller = document.querySelector('#content-body');
            if (scroller) {
                scroller.scrollBy(0, scroller.clientHeight * 0.9);
            } else {
                window.scrollBy(0, window.innerHeight * 0.9);
            }
        """)
        time.sleep(SCROLL_SLEEP)

    # 키워드 포함 카드만 필터링
    filtered = []
    for eid, text in collected.items():
        t_norm = text.replace(" ", "")
        if kw in t_norm:
            filtered.append({"event_id": eid, "card_text": text})

    return filtered


# =========================
# 메인 로직
# =========================

def crawl_babitalk_event_urls():
    df_cat = pd.read_excel(INPUT_EXCEL)
    print(f"📂 카테고리 로드: {df_cat.shape[0]} rows\n")

    has_big = COL_BIG in df_cat.columns
    has_mid = COL_MID in df_cat.columns

    if COL_SMALL not in df_cat.columns:
        raise ValueError(f"엑셀에 '{COL_SMALL}' 컬럼이 없습니다. 컬럼명을 확인해 주세요.")

    driver = setup_driver(headless=HEADLESS)
    all_rows = []

    for idx, row in df_cat.iterrows():
        small = str(row[COL_SMALL]).strip()
        if not small:
            continue

        big = str(row[COL_BIG]).strip() if has_big else None
        mid = str(row[COL_MID]).strip() if has_mid else None

        q = quote_plus(small)
        search_url = f"{BABITALK_SEARCH_BASE}?keyword={q}&tab=events"

        print(f"\n=== [{idx+1}/{len(df_cat)}] {big} / {mid} / {small} ===")
        print("🔍 검색 URL:", search_url)

        # 1) 검색 페이지 로드
        loaded = False
        for attempt in range(3):
            try:
                driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
                driver.get(search_url)
                time.sleep(WAIT_AFTER_LOAD)
                loaded = True
                break
            except TimeoutException:
                print(f"⏰ 검색 페이지 로드 지연 — 재시도 {attempt+1}/3")
                try:
                    driver.execute_script("window.stop();")
                except Exception:
                    pass
                time.sleep(2)

        if not loaded:
            print("🚫 검색 페이지 로드 실패 → 이 소분류 스킵")
            continue

        # 2) "결과 없음" 페이지면 바로 스킵
        if is_no_result_page_babitalk(driver):
            print("   ❌ 결과가 없습니다. → 이 키워드 스킵")
            continue

        # 3) 실제 검색결과가 있는 경우만 카드 수집
        cards = collect_babitalk_event_cards(driver, small)
        print(f"   🎯 '{small}' 키워드 포함 바비톡 이벤트: {len(cards)}개")

        if not cards:
            continue

        for c in cards:
            eid = c["event_id"]
            event_url = f"https://web.babitalk.com/events/{eid}"

            all_rows.append({
                "대분류": big,
                "중분류": mid,
                "소분류": small,
                "babitalk_search_url": search_url,
                "event_id": eid,
                "event_url": event_url,
                "card_text": c["card_text"],
            })

        # (선택) 20개마다 중간 저장
        if (idx + 1) % 20 == 0:
            tmp = pd.DataFrame(all_rows)
            tmp.to_excel("babitalk_event_urls_tmp.xlsx",
                         index=False, engine="openpyxl")
            print("💾 중간 저장: babitalk_event_urls_tmp.xlsx")

    driver.quit()

    df_out = pd.DataFrame(all_rows)
    df_out.to_excel(OUTPUT_EXCEL, index=False)
    print(f"\n✅ 완료 → {OUTPUT_EXCEL} (총 {df_out.shape[0]}행)")

    return df_out


if __name__ == "__main__":
    crawl_babitalk_event_urls()

📂 카테고리 로드: 622 rows


=== [1/622] 리프팅 / 레이저리프팅 / 울쎄라리프팅 ===
🔍 검색 URL: https://web.babitalk.com/search?keyword=%EC%9A%B8%EC%8E%84%EB%9D%BC%EB%A6%AC%ED%94%84%ED%8C%85&tab=events
   ▸ Scroll 001 / cards: 19 (+19)
   ▸ Scroll 002 / cards: 19 (+0)
   ▸ Scroll 003 / cards: 19 (+0)
   ▸ Scroll 004 / cards: 19 (+0)
   ▸ Scroll 005 / cards: 19 (+0)
   ▸ Scroll 006 / cards: 19 (+0)
   ▸ Scroll 007 / cards: 19 (+0)
   ✅ 더 이상 새 카드 없음 → 스크롤 중단
   🎯 '울쎄라리프팅' 키워드 포함 바비톡 이벤트: 19개

=== [2/622] 리프팅 / 레이저리프팅 / 인모드리프팅 ===
🔍 검색 URL: https://web.babitalk.com/search?keyword=%EC%9D%B8%EB%AA%A8%EB%93%9C%EB%A6%AC%ED%94%84%ED%8C%85&tab=events
   ▸ Scroll 001 / cards: 24 (+24)
   ▸ Scroll 002 / cards: 24 (+0)
   ▸ Scroll 003 / cards: 24 (+0)
   ▸ Scroll 004 / cards: 24 (+0)
   ▸ Scroll 005 / cards: 24 (+0)
   ▸ Scroll 006 / cards: 44 (+20)
   ▸ Scroll 007 / cards: 44 (+0)
   ▸ Scroll 008 / cards: 44 (+0)
   ▸ Scroll 009 / cards: 44 (+0)
   ▸ Scroll 010 / cards: 44 (+0)
   ▸ Scroll 011 / cards: 44 (+0)
   ▸ Scroll